# One-at-a-Time (OAT) Manning's n Sensitivity Analysis

This notebook performs a **one-at-a-time (OAT)** sensitivity analysis on Manning's n values
using a 2D HEC-RAS model. Each land cover class is varied individually while holding all
others at their baseline values, producing sensitivity curves and a tornado diagram.

**Key Concepts:**
- OAT sensitivity: vary one parameter at a time to isolate its effect
- Manning's n range tables with configurable interval stepping
- Canonical roughness propagation: cloned geometry, plaintext `LCMann Table` edits, and fresh geometry preprocessing
- Distributed plan execution with RasRemote workers, falling back to local `RasCmdr.compute_parallel()`
- Sensitivity curves and tornado diagram visualization
- Spatial/global delta metrics for max WSE, inundation volume, face velocity, and arrival time

**Version note:** the Sabinal ScienceBase validation run exposed a HEC-RAS 6.6 DSS finalization
crash (`RasUnsteady.exe ... DSS 91 DSS.for`) after hydraulics completed. HEC-RAS 7.0 completed the
same p10/p14/p20/p54/p59/p65 plans cleanly with full final HDFs, so this notebook defaults to 7.0.

**Prerequisites:** HEC-RAS 7.0+ installed, ras-commander package, optional RasRemote worker JSON.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ras_commander import (
        init_ras_project, RasCmdr, RasPlan, RasGeo, RasExamples,
        RasMap, RasMonteCarlo, RasCurrency
    )
    from ras_commander.hdf import HdfMesh, HdfResultsMesh, HdfResultsPlan, HdfLandCover
except ImportError:
    current_file = Path("__file__").resolve()
    parent_directory = current_file.parent.parent
    sys.path.append(str(parent_directory))
    from ras_commander import (
        init_ras_project, RasCmdr, RasPlan, RasGeo, RasExamples,
        RasMap, RasMonteCarlo, RasCurrency
    )
    from ras_commander.hdf import HdfMesh, HdfResultsMesh, HdfResultsPlan, HdfLandCover

## Configuration

All parameters for the analysis are defined here. Adjust `INTERVAL` to control
the granularity of the sensitivity sweep (smaller interval = more plans = longer runtime).

In [ ]:
# === PROJECT CONFIGURATION ===
PROJECT_NAME = os.getenv("RAS_COMMANDER_711_PROJECT_NAME", "Muncie")  # RasExamples project name
PROJECT_PATH_OVERRIDE = os.getenv("RAS_COMMANDER_711_PROJECT_PATH")    # Optional external RAS project folder
RAS_VERSION = os.getenv("RAS_COMMANDER_711_RAS_VERSION", "7.0")        # HEC-RAS version string
TEMPLATE_PLAN = os.getenv("RAS_COMMANDER_711_TEMPLATE_PLAN", "04")    # Plan number with 2D geometry

# === SENSITIVITY CONFIGURATION ===
INTERVAL = float(os.getenv("RAS_COMMANDER_711_INTERVAL", "0.02"))      # Manning's n step size for sweep

# === POINT OF INTEREST ===
# Defaults are for the Muncie example project (State Plane Indiana East, ft).
# For external projects, override with RAS_COMMANDER_711_POI_X/Y/LABEL.
POI_X = float(os.getenv("RAS_COMMANDER_711_POI_X", "408350.0"))
POI_Y = float(os.getenv("RAS_COMMANDER_711_POI_Y", "1802550.0"))
POI_LABEL = os.getenv("RAS_COMMANDER_711_POI_LABEL", "Mid-Floodplain POI")

# === EXECUTION ===
MAX_WORKERS = int(os.getenv("RAS_COMMANDER_711_MAX_WORKERS", "4"))     # Local parallel workers
NUM_CORES = int(os.getenv("RAS_COMMANDER_711_NUM_CORES", "2"))         # CPU cores per HEC-RAS instance

# === RASREMOTE EXECUTION ===
def _env_bool(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

USE_REMOTE_WORKERS = _env_bool("RAS_COMMANDER_711_USE_REMOTE", True)
REQUESTED_REMOTE_WORKERS = tuple(
    label.strip()
    for label in os.getenv("RAS_COMMANDER_711_REMOTE_WORKERS", "CLB02,CLB04,CLB06,CLB10").split(",")
    if label.strip()
)
CLEAR_GEOMPRE = _env_bool(
    "RAS_COMMANDER_711_CLEAR_GEOMPRE", True
)  # Regenerate geometry preprocessor files after Manning's n edits
FORCE_RERUN = _env_bool("RAS_COMMANDER_711_FORCE_RERUN", True)
REUSE_EXISTING_OAT_PLANS = _env_bool("RAS_COMMANDER_711_REUSE_EXISTING_OAT_PLANS", False)
RUN_HECRAS = _env_bool("RAS_COMMANDER_711_RUN_HECRAS", True)
REPAIR_GEOM_ASSOCIATIONS = _env_bool(
    "RAS_COMMANDER_711_REPAIR_GEOM_ASSOCIATIONS", True
)  # Ensure cloned geometry HDFs retain terrain/land-cover associations


## Configure RasRemote Workers

Load the CLB RasRemote fleet from the gitignored worker configuration. Passwords are never printed.


In [ ]:
import json

# RasRemote Worker Setup
# Loads a worker fleet from working/remote_workers_clb.json (gitignored, has real credentials).
# Falls back to examples/remote_workers_clb_template.json (committed, redacted) for reference.
# Requested labels are filtered so this notebook cannot spill onto older CLB workers.
_cwd = Path.cwd()
if (_cwd / "remote_workers_clb_template.json").exists():
    _repo_root = _cwd.parent
    _examples_dir = _cwd
else:
    _repo_root = _cwd
    _examples_dir = _cwd / "examples"

_workers_json_real = _repo_root / "working" / "remote_workers_clb.json"
_workers_json_template = _examples_dir / "remote_workers_clb_template.json"
_requested_labels = {label.upper() for label in REQUESTED_REMOTE_WORKERS}

def _worker_label_from_name(name: str):
    normalized = name.upper().replace("CLB-", "CLB")
    for label in sorted(_requested_labels):
        if label in normalized:
            return label
    return None

remote_workers = []
remote_worker_status = pd.DataFrame()

if USE_REMOTE_WORKERS:
    try:
        from ras_commander.remote import load_workers_from_json

        _cfg_path = _workers_json_real if _workers_json_real.exists() else _workers_json_template
        _cfg_raw = json.loads(_cfg_path.read_text())

        _status_rows = []
        _enabled_requested_labels = set()
        for _entry in _cfg_raw.get("workers", []):
            _label = _worker_label_from_name(_entry.get("name", ""))
            if _label is None:
                continue
            if _entry.get("enabled", False):
                _enabled_requested_labels.add(_label)
            _status_rows.append({
                "label": _label,
                "name": _entry.get("name", ""),
                "enabled": bool(_entry.get("enabled", False)),
                "hostname": _entry.get("hostname", ""),
                "share_path": _entry.get("share_path", ""),
                "session_id": _entry.get("session_id", ""),
                "slots": (_entry.get("cores_total", 0) or 0) // max((_entry.get("cores_per_plan", 1) or 1), 1),
                "note": _entry.get("_note", ""),
            })

        _loaded_workers = load_workers_from_json(_cfg_path, enabled_only=True)
        remote_workers = [
            _worker for _worker in _loaded_workers
            if getattr(_worker, "worker_type", "") != "local"
            and _worker_label_from_name(getattr(_worker, "worker_id", "")) in _requested_labels
        ]

        _loaded_labels = {
            _worker_label_from_name(getattr(_worker, "worker_id", ""))
            for _worker in remote_workers
        }
        _loaded_labels.discard(None)
        _missing_labels = sorted(_requested_labels - _loaded_labels)
        _total_slots = sum(getattr(_worker, "max_parallel_plans", 1) or 1 for _worker in remote_workers)

        remote_worker_status = pd.DataFrame(_status_rows).sort_values(["label", "name"])
        print(remote_worker_status.to_string(index=False))
        print(f"Loaded {len(remote_workers)} RasRemote workers ({_total_slots} concurrent slots).")
        if _missing_labels:
            print(f"Requested worker labels not loaded: {', '.join(_missing_labels)}")
    except Exception as _e:
        print(f"Warning: could not load RasRemote workers ({_e}) -- falling back to local execution.")
        USE_REMOTE_WORKERS = False
        remote_workers = []
else:
    print("USE_REMOTE_WORKERS = False -- using local compute_parallel.")
    print(f"Worker config available at: {_workers_json_real}")


## Extract Example Project and Initialize

In [ ]:
if PROJECT_PATH_OVERRIDE:
    project_folder = Path(PROJECT_PATH_OVERRIDE).expanduser()
    project_source = "external override"
else:
    project_folder = RasExamples.extract_project(PROJECT_NAME)
    project_source = "RasExamples"

ras = init_ras_project(project_folder, RAS_VERSION)

print(f"Project: {project_folder}")
print(f"Source: {project_source}")
print(f"RAS version: {RAS_VERSION}")
print(f"Plans: {len(ras.plan_df)}")
print(f"Template plan: {TEMPLATE_PLAN}")
print()
print("Plan DataFrame:")
ras.plan_df[["plan_number", "Plan Title", "Geom File"]].head(10)


## Read Baseline Manning’s n Values

Extract the current Manning’s n values from the template geometry file.
These serve as the baseline for the OAT sensitivity analysis.

In [ ]:
# Get geometry file path from template plan
template_row = ras.plan_df[ras.plan_df["plan_number"] == TEMPLATE_PLAN].iloc[0]
template_geom_number = str(template_row["geometry_number"]).zfill(2)
geom_file = Path(template_row["Geom Path"])
if not geom_file.exists():
    geom_file = project_folder / f"{ras.project_name}.g{template_row['geometry_number']}"

# Read base overrides (primary Manning's n table)
base_overrides_df = RasGeo.get_mannings_baseoverrides(geom_file).copy()
base_overrides_df["Base Mannings n Value"] = pd.to_numeric(
    base_overrides_df["Base Mannings n Value"], errors="coerce"
)
base_overrides = dict(zip(
    base_overrides_df["Land Cover Name"],
    base_overrides_df["Base Mannings n Value"],
))

print("Base Manning's n Overrides:")
print(f"  Land covers: {len(base_overrides)}")
for name, n_val in base_overrides.items():
    print(f"  {name}: n = {n_val}")

# Read regional overrides
regional_overrides_df = RasGeo.get_mannings_regionoverrides(geom_file)
print()
print(f"Regional Overrides: {len(regional_overrides_df)} rows")
if not regional_overrides_df.empty:
    print(regional_overrides_df.to_string(index=False))


## Build Manning’s n Range Table

Define min/max bounds for each land cover class. Obstruction values (n >= 1.0,
typically buildings at n=100) are excluded from the sensitivity analysis since
they represent physical barriers, not roughness.

In [ ]:
# Standard Manning's n ranges by land cover type
# Source: HEC-RAS Reference Manual, Chow (1959)
MANNINGS_RANGES = {
    "Pasture":           (0.025, 0.050),
    "Brush":             (0.040, 0.120),
    "Trees":             (0.060, 0.200),
    "Low Intensity":     (0.040, 0.120),
    "High Intensity":    (0.060, 0.200),
    "Water":             (0.025, 0.050),
    "Cropland":          (0.020, 0.060),
    "Wetland":           (0.030, 0.100),
    "Barren":            (0.010, 0.035),
}

OBSTRUCTION_THRESHOLD = 1.0  # n values >= this are obstructions

# Build range table from base overrides
range_table = {}
excluded = {}

for name, baseline_n in base_overrides.items():
    if baseline_n >= OBSTRUCTION_THRESHOLD:
        excluded[name] = baseline_n
        continue
    
    if name in MANNINGS_RANGES:
        n_min, n_max = MANNINGS_RANGES[name]
    else:
        # Default: +/- 50% of baseline, clamped to [0.01, 0.5]
        n_min = max(0.01, baseline_n * 0.5)
        n_max = min(0.50, baseline_n * 1.5)
    
    range_table[name] = {
        "baseline": baseline_n,
        "min": n_min,
        "max": n_max,
        "steps": np.arange(n_min, n_max + INTERVAL / 2, INTERVAL)
    }

print("=== Sensitivity Range Table ===")
print(f"Variable land covers: {len(range_table)}")
print(f"Excluded (obstructions): {len(excluded)}")
print()

total_plans = sum(len(v["steps"]) for v in range_table.values())
print(f"Total plans to create: {total_plans}")
print()

for name, info in range_table.items():
    print(f'  {name}: baseline={info["baseline"]:.3f}, '
          f'range=[{info["min"]:.3f}, {info["max"]:.3f}], '
          f'steps={len(info["steps"])}')

if excluded:
    print(f"\nExcluded obstructions:")
    for name, n_val in excluded.items():
        print(f"  {name}: n = {n_val} (held constant)")

## Create OAT Sensitivity Plans

For each land cover, create a set of plans that vary only that land cover's
Manning's n while keeping all others at baseline. Each plan gets a cloned
geometry file, and the cloned plan is modified through
`RasMonteCarlo.make_mannings_apply_fn(path="plaintext")`, which edits the
plain-text `LCMann Table` in that clone's `.g##`. Execution must use
`clear_geompre=True` so HEC-RAS regenerates cached per-cell Manning's n.


In [ ]:
import re

plan_registry = []  # Track all plans used for this analysis
plan_counter = 0
plan_registry_csv = project_folder / "oat_plan_registry.csv"

# Canonical 2D land-cover roughness propagation path:
#   cloned .g## per plan + plaintext LCMann Table edit + clear_geompre=True at execution.
# This mirrors RasMonteCarlo.run_ensemble(..., clone_geom=True, clear_geompre=True)
# without building a stochastic ensemble.
mannings_apply_fn = RasMonteCarlo.make_mannings_apply_fn(
    zone_column_map={str(name): str(name) for name in base_overrides_df["Land Cover Name"]},
    path="plaintext",
)


def _display_n_value(value):
    # Plan titles carry three decimals, so the registry also stores the title
    # precision for stable reuse across projects that already have OAT plans.
    return round(float(f"{float(value):.3f}"), 4)


def _expected_oat_keys():
    keys = set()
    for lc_name, info in range_table.items():
        for n_value in info["steps"]:
            keys.add((str(lc_name), _display_n_value(n_value)))
    return keys


def _parse_oat_title(title):
    match = re.match(r"^OAT_(?P<land_cover>.+?)_(?P<n>[0-9.]+)$", str(title).strip())
    if not match:
        return None
    return str(match.group("land_cover")), _display_n_value(match.group("n"))


def _normalize_registry_df(df, source):
    required = {"land_cover", "n_value", "plan_number", "geom_number", "Plan Title", "is_baseline"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(f"Plan registry {source} is missing required column(s): {sorted(missing)}")
    normalized = df.copy()
    normalized["land_cover"] = normalized["land_cover"].astype(str)
    normalized["n_value"] = normalized["n_value"].map(_display_n_value)
    normalized["plan_number"] = normalized["plan_number"].astype(str).str.zfill(2)
    normalized["geom_number"] = normalized["geom_number"].astype(str).str.zfill(2)
    normalized["is_baseline"] = normalized["is_baseline"].astype(bool)
    normalized["registry_source"] = source
    return normalized


def _registry_from_plan_titles():
    expected_keys = _expected_oat_keys()
    parsed_rows = []
    for _, plan_row in ras.plan_df.iterrows():
        parsed = _parse_oat_title(plan_row.get("Plan Title", ""))
        if parsed is None or parsed not in expected_keys:
            continue
        lc_name, n_value = parsed
        baseline_n = float(range_table[lc_name]["baseline"])
        parsed_rows.append({
            "land_cover": lc_name,
            "n_value": n_value,
            "plan_number": str(plan_row["plan_number"]).zfill(2),
            "geom_number": str(plan_row.get("geometry_number", "")).zfill(2),
            "Plan Title": str(plan_row.get("Plan Title", "")),
            "is_baseline": abs(float(n_value) - baseline_n) < 1e-6,
        })
    return pd.DataFrame(parsed_rows)


if REUSE_EXISTING_OAT_PLANS:
    expected_keys = _expected_oat_keys()
    print(f"Reusing existing OAT plans from project ({len(expected_keys)} expected plan keys)...")
    if plan_registry_csv.exists():
        print(f"Loading explicit OAT registry: {plan_registry_csv}")
        registry_df = _normalize_registry_df(pd.read_csv(plan_registry_csv), str(plan_registry_csv))
    else:
        print("No explicit OAT registry found; falling back to OAT_* plan-title parsing.")
        registry_df = _normalize_registry_df(_registry_from_plan_titles(), "plan_titles")

    if registry_df.empty:
        raise RuntimeError("REUSE_EXISTING_OAT_PLANS=True, but no matching OAT plans were found.")
    registry_df = (
        registry_df.sort_values(["land_cover", "n_value", "plan_number"])
        .drop_duplicates(["land_cover", "n_value"], keep="first")
        .reset_index(drop=True)
    )
    found_keys = set(zip(registry_df["land_cover"].astype(str), registry_df["n_value"].round(4)))
    missing_keys = sorted(expected_keys - found_keys)
    if missing_keys:
        raise RuntimeError(f"Missing {len(missing_keys)} expected OAT plan(s): {missing_keys[:10]}")
else:
    for lc_name, info in range_table.items():
        print(f"\nCreating plans for: {lc_name} ({len(info['steps'])} steps)")

        for n_value in info["steps"]:
            plan_counter += 1
            registry_n = _display_n_value(n_value)
            plan_title = f"OAT_{str(lc_name)[:8]}_{registry_n:.3f}"

            # Clone plan and geometry from template. Each sensitivity plan needs its own
            # geometry so its plaintext LCMann Table does not collide with other runs.
            new_plan_number = RasPlan.clone_plan(TEMPLATE_PLAN, ras_object=ras)
            new_geom_number = RasPlan.clone_geom(template_geom_number, ras_object=ras)

            # Link cloned plan to cloned geometry
            RasPlan.set_geom(new_plan_number, new_geom_number, ras_object=ras)

            # Set plan title
            plan_file = ras.project_folder / f"{ras.project_name}.p{new_plan_number}"
            RasPlan.set_plan_title(plan_file, plan_title)

            # Modify Manning's n through the hardened plaintext LCMann propagation path:
            # build a full row of baseline class n values, then perturb only the target class.
            sample_row = pd.Series({str(k): float(v) for k, v in base_overrides.items()})
            sample_row[str(lc_name)] = float(n_value)
            mannings_apply_fn(plan_file, sample_row, ras_object=ras)

            plan_registry.append({
                "land_cover": str(lc_name),
                "n_value": registry_n,
                "plan_number": str(new_plan_number).zfill(2),
                "geom_number": str(new_geom_number).zfill(2),
                "Plan Title": plan_title,
                "is_baseline": abs(float(n_value) - float(info["baseline"])) < 1e-6,
                "registry_source": "created",
            })

    # Refresh project state after cloning plans and geometries.
    ras = init_ras_project(project_folder, RAS_VERSION)
    registry_df = pd.DataFrame(plan_registry)

registry_df.to_csv(plan_registry_csv, index=False)
print(f"OAT plan registry: {plan_registry_csv}")

print("\n=== Plan Registry ===")
print(f"Total plans registered: {len(registry_df)}")
print(f"Land covers varied: {registry_df['land_cover'].nunique()}")
print("\nPlans per land cover:")
print(registry_df.groupby("land_cover").size())

## Verify Cloned Geometry Associations

Each OAT plan uses a cloned geometry. The plain-text `.g##` file stores the `LCMann Table`, but the compiled `.g##.hdf` stores the terrain and land-cover sidecar associations HEC-RAS uses during geometry preprocessing. This QA step repairs cloned geometry HDF associations to the template geometry before any hydraulic compute.


In [ ]:
def _same_resolved_path(left, right):
    if left in (None, "") or right in (None, ""):
        return False
    return os.path.normcase(os.path.abspath(str(left))) == os.path.normcase(os.path.abspath(str(right)))


def _first_existing_path(df, preferred_name=None):
    if df is None or df.empty or "resolved_path" not in df.columns:
        return None
    candidates = df.copy()
    if "checked" in candidates.columns:
        checked = candidates[candidates["checked"].astype(bool)]
        if not checked.empty:
            candidates = checked
    if preferred_name and "name" in candidates.columns:
        exact = candidates[candidates["name"].astype(str).str.lower() == preferred_name.lower()]
        if not exact.empty:
            candidates = exact
    for value in candidates["resolved_path"].dropna():
        candidate = Path(value)
        if candidate.exists():
            return candidate
    return None


def _geom_hdf_for_plan(plan_number, geom_number):
    hdf_path = RasCurrency.get_geom_hdf_path(str(plan_number).zfill(2), ras)
    if hdf_path is not None:
        return Path(hdf_path)
    return ras.project_folder / f"{ras.project_name}.g{str(geom_number).zfill(2)}.hdf"


template_geom_hdf = RasCurrency.get_geom_hdf_path(TEMPLATE_PLAN, ras)
if template_geom_hdf is None:
    raise RuntimeError(f"Could not resolve template geometry HDF for plan {TEMPLATE_PLAN}.")
template_geom_hdf = Path(template_geom_hdf)

template_assoc = RasMap.get_hdf_geometry_association(template_geom_hdf)
source_landcover_hdf = template_assoc.get("landcover_hdf_path")
source_terrain_hdf = template_assoc.get("terrain_hdf_path")

if source_landcover_hdf is None:
    landcover_layers = RasMap.list_landcover_layers(project_folder)
    source_landcover_hdf = _first_existing_path(landcover_layers, preferred_name="Mannings_n")
if source_terrain_hdf is None:
    terrain_layers = RasMap.list_terrain_layers(project_folder)
    source_terrain_hdf = _first_existing_path(terrain_layers, preferred_name="terrain_9")

if source_landcover_hdf is None or not Path(source_landcover_hdf).exists():
    raise RuntimeError(
        "No land-cover HDF association could be resolved from the template geometry "
        "or RASMapper. Use the provided Mannings_n layer; do not reauthor rasters."
    )
if source_terrain_hdf is None or not Path(source_terrain_hdf).exists():
    raise RuntimeError("No terrain HDF association could be resolved for cloned geometries.")

source_landcover_hdf = Path(source_landcover_hdf)
source_terrain_hdf = Path(source_terrain_hdf)
association_rows = []

for _, row in (
    registry_df[["plan_number", "geom_number", "Plan Title"]]
    .drop_duplicates(["geom_number"])
    .sort_values("geom_number")
    .iterrows()
):
    geom_hdf = _geom_hdf_for_plan(row["plan_number"], row["geom_number"])
    before = {}
    if geom_hdf.exists():
        before = RasMap.get_hdf_geometry_association(geom_hdf)
    needs_repair = (
        not geom_hdf.exists()
        or not _same_resolved_path(before.get("landcover_hdf_path"), source_landcover_hdf)
        or not _same_resolved_path(before.get("terrain_hdf_path"), source_terrain_hdf)
    )

    repaired = False
    if needs_repair:
        if not REPAIR_GEOM_ASSOCIATIONS:
            association_rows.append({
                "plan_number": str(row["plan_number"]).zfill(2),
                "geom_number": str(row["geom_number"]).zfill(2),
                "geom_hdf": geom_hdf.name,
                "status": "needs_repair_disabled",
                "association_ok": False,
                "landcover_raw_filename": before.get("landcover_raw_filename", ""),
                "terrain_raw_filename": before.get("terrain_raw_filename", ""),
            })
            continue
        RasMap.associate_geometry_layers(
            project_folder,
            geom_hdf,
            landcover_hdf_path=source_landcover_hdf,
            terrain_hdf_path=source_terrain_hdf,
            ras_object=ras,
        )
        repaired = True

    after = RasMap.get_hdf_geometry_association(geom_hdf)
    ok = (
        _same_resolved_path(after.get("landcover_hdf_path"), source_landcover_hdf)
        and _same_resolved_path(after.get("terrain_hdf_path"), source_terrain_hdf)
    )
    association_rows.append({
        "plan_number": str(row["plan_number"]).zfill(2),
        "geom_number": str(row["geom_number"]).zfill(2),
        "geom_hdf": geom_hdf.name,
        "status": "repaired" if repaired else "ok",
        "association_ok": ok,
        "landcover_raw_filename": after.get("landcover_raw_filename", ""),
        "landcover_layer_name": after.get("landcover_layer_name", ""),
        "terrain_raw_filename": after.get("terrain_raw_filename", ""),
        "terrain_layer_name": after.get("terrain_layer_name", ""),
    })

association_qaqc_df = pd.DataFrame(association_rows)
display(association_qaqc_df)

association_ok = association_qaqc_df["association_ok"].fillna(False).astype(bool)
bad_associations = association_qaqc_df[~association_ok]
if not bad_associations.empty:
    raise RuntimeError(
        f"{len(bad_associations)} cloned geometry HDF association(s) are not bound to "
        f"{source_landcover_hdf.name}. Repair these before running hydraulics."
    )

print(
    "Geometry association QA complete: "
    f"{int((association_qaqc_df['status'] == 'repaired').sum())} repaired, "
    f"{int((association_qaqc_df['status'] == 'ok').sum())} already OK. "
    f"Land cover: {source_landcover_hdf.name}; terrain: {source_terrain_hdf.name}."
)


## Execute All Plans

Run the sensitivity plans with RasRemote when configured, otherwise use local parallel execution.


In [ ]:
all_plan_numbers = registry_df["plan_number"].astype(str).str.zfill(2).tolist()

if not RUN_HECRAS:
    print(f"RUN_HECRAS=False -- skipping compute and verifying {len(all_plan_numbers)} existing HDFs.")
    execution_summary = pd.DataFrame({
        "plan_number": all_plan_numbers,
        "success": True,
        "execution_mode": "existing_hdf_postprocess",
    })
elif USE_REMOTE_WORKERS and remote_workers:
    from ras_commander.remote import compute_parallel_remote

    total_slots = sum(getattr(worker, "max_parallel_plans", 1) or 1 for worker in remote_workers)
    print(
        f"Executing {len(all_plan_numbers)} plans with RasRemote "
        f"on {len(remote_workers)} workers ({total_slots} slots)..."
    )

    remote_results = compute_parallel_remote(
        plan_numbers=all_plan_numbers,
        workers=remote_workers,
        ras_object=ras,
        num_cores=NUM_CORES,
        clear_geompre=CLEAR_GEOMPRE,
        force_rerun=FORCE_RERUN,
    )

    missing_remote = sorted(set(all_plan_numbers) - set(remote_results.keys()))
    if missing_remote:
        raise RuntimeError(f"RasRemote returned no result for plan(s): {missing_remote}")

    execution_summary = pd.DataFrame([
        {
            "plan_number": plan_num,
            "worker_id": result.worker_id,
            "success": result.success,
            "execution_time_sec": round(result.execution_time, 1),
            "hdf_path": result.hdf_path or "",
            "error_message": result.error_message or "",
        }
        for plan_num, result in sorted(remote_results.items())
    ])
    print(execution_summary.to_string(index=False))

    failed = execution_summary[~execution_summary["success"]]
    if not failed.empty:
        raise RuntimeError(f"{len(failed)} RasRemote plan(s) failed; see execution_summary above.")
else:
    print(f"Executing {len(all_plan_numbers)} plans locally with {MAX_WORKERS} workers...")
    local_result = RasCmdr.compute_parallel(
        plan_number=all_plan_numbers,
        max_workers=MAX_WORKERS,
        num_cores=NUM_CORES,
        clear_geompre=CLEAR_GEOMPRE,
        force_rerun=FORCE_RERUN,
        verify=True,
        ras_object=ras,
    )
    execution_summary = local_result.results_df.copy()
    failed_local = sorted(
        plan_num for plan_num in all_plan_numbers
        if not bool(local_result.execution_results.get(plan_num, False))
    )
    if failed_local:
        raise RuntimeError(f"{len(failed_local)} local plan(s) failed verification: {failed_local}")

# Refresh after execution and explicitly check the final HDFs. This catches the
# Sabinal failure mode where hydraulics finished but HEC-RAS 6.6 crashed while
# writing DSS output, leaving a tiny final HDF plus a full .tmp.hdf.
ras = init_ras_project(project_folder, RAS_VERSION)
post_run_rows = []
for plan_num in all_plan_numbers:
    hdf_path = RasCurrency.get_plan_hdf_path(plan_num, ras)
    hdf_path = Path(hdf_path) if hdf_path is not None else None
    exists = bool(hdf_path and hdf_path.exists())
    complete = bool(exists and RasCurrency.check_plan_hdf_complete(hdf_path))
    message_flags = {
        "writing_results_to_dss": False,
        "dss_for": False,
        "error_with_program": False,
        "finished_unsteady": False,
        "complete_process": False,
    }
    if exists:
        try:
            msgs = HdfResultsPlan.get_compute_messages_hdf_only(hdf_path)
            message_flags = {
                "writing_results_to_dss": "Writing Results to DSS" in msgs,
                "dss_for": "DSS.for" in msgs,
                "error_with_program": "Error with program" in msgs,
                "finished_unsteady": "Finished Unsteady Flow Simulation" in msgs,
                "complete_process": "Complete Process" in msgs,
            }
        except Exception as exc:
            message_flags["message_read_error"] = str(exc)

    post_run_rows.append({
        "plan_number": plan_num,
        "hdf_path": str(hdf_path) if hdf_path else "",
        "hdf_exists": exists,
        "hdf_complete": complete,
        "hdf_size_bytes": hdf_path.stat().st_size if exists else 0,
        **message_flags,
    })

post_run_qaqc_df = pd.DataFrame(post_run_rows)
display(post_run_qaqc_df)

bad_post_run = post_run_qaqc_df[
    (~post_run_qaqc_df["hdf_complete"])
    | post_run_qaqc_df["dss_for"]
    | post_run_qaqc_df["error_with_program"]
]
if not bad_post_run.empty:
    display(bad_post_run)
    raise RuntimeError(
        f"{len(bad_post_run)} plan(s) failed post-run HDF/DSS QAQC. "
        "Use HEC-RAS 7.0+ for Sabinal-style DSS finalization failures."
    )

print("Execution/post-processing readiness check complete; all final HDFs passed QAQC.")


## Verify Roughness Propagation

Before interpreting WSE differences, verify that at least one OAT roughness change reached the
preprocessed 2D geometry. This checks the cached `Cells Center Manning's n` dataset in the
cloned geometry HDFs. If this fails, the runs are hydraulically inert regardless of HEC-RAS
completion status. Converted legacy projects may still fail here if the land-cover association
cannot be sampled into 2D cells.


In [ ]:
propagation_rows = []
for lc_name, group in registry_df.groupby("land_cover"):
    if len(group) < 2:
        continue
    low_row = group.sort_values("n_value").iloc[0]
    high_row = group.sort_values("n_value").iloc[-1]
    low_hdf = RasCurrency.get_geom_hdf_path(low_row["plan_number"], ras)
    high_hdf = RasCurrency.get_geom_hdf_path(high_row["plan_number"], ras)
    if low_hdf is None or high_hdf is None:
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": "",
            "high_geom": "",
            "status": "missing_geom_hdf_path",
            "max_delta_n": np.nan,
        })
        continue

    low_df = HdfLandCover.get_preprocessed_mannings_n(low_hdf)
    high_df = HdfLandCover.get_preprocessed_mannings_n(high_hdf)
    if low_df.empty or high_df.empty:
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": Path(low_hdf).name,
            "high_geom": Path(high_hdf).name,
            "status": "missing_cell_mannings_n",
            "max_delta_n": np.nan,
        })
        continue

    merged = low_df.merge(
        high_df,
        on=["mesh_name", "cell_id"],
        how="outer",
        suffixes=("_low", "_high"),
        indicator=True,
    )
    if not (merged["_merge"] == "both").all():
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": Path(low_hdf).name,
            "high_geom": Path(high_hdf).name,
            "status": "mesh_cell_mismatch",
            "max_delta_n": np.nan,
            "low_cells": int(len(low_df)),
            "high_cells": int(len(high_df)),
            "unmatched_cells": int((merged["_merge"] != "both").sum()),
        })
        continue

    delta = np.abs(
        merged["mannings_n_high"].to_numpy(dtype=float)
        - merged["mannings_n_low"].to_numpy(dtype=float)
    )
    propagation_rows.append({
        "land_cover": lc_name,
        "low_geom": Path(low_hdf).name,
        "high_geom": Path(high_hdf).name,
        "low_n": float(low_row["n_value"]),
        "high_n": float(high_row["n_value"]),
        "status": "ok",
        "mesh_count": int(merged["mesh_name"].nunique()),
        "cell_count": int(len(merged)),
        "max_delta_n": float(np.nanmax(delta)),
        "changed_cells": int(np.count_nonzero(delta > 1e-8)),
    })

propagation_df = pd.DataFrame(propagation_rows)
display(propagation_df.sort_values("max_delta_n", ascending=False, na_position="last"))

valid_delta = propagation_df.loc[propagation_df["status"] == "ok", "max_delta_n"].dropna()
failed_propagation = propagation_df[
    (propagation_df["status"] != "ok")
    | (pd.to_numeric(propagation_df["max_delta_n"], errors="coerce").fillna(0.0) <= 1e-8)
]
if valid_delta.empty or not failed_propagation.empty:
    display(failed_propagation)
    raise RuntimeError(
        "Manning's n perturbations did not reach preprocessed cell n for every varied class. "
        "Verify cloned geometry HDF associations, plaintext LCMann edits, and clear_geompre=True."
    )

print(
    f"Roughness propagation confirmed for {len(valid_delta)} land-cover sweep(s): "
    f"max |delta n| = {float(valid_delta.max()):.6g}"
)


## Extract Results at Point of Interest

For each executed plan, extract the maximum water surface elevation (WSE)
at the point of interest. This allows us to measure the sensitivity of WSE
to each land cover’s Manning’s n value.

In [ ]:
from scipy.spatial import cKDTree

results = []
extraction_failures = []

for _, row in registry_df.iterrows():
    plan_num = row["plan_number"]

    try:
        hdf_path = RasCurrency.get_plan_hdf_path(plan_num, ras)
        if hdf_path is None or not Path(hdf_path).exists():
            extraction_failures.append({
                "plan_number": plan_num,
                "land_cover": row["land_cover"],
                "n_value": row["n_value"],
                "reason": "missing_hdf_results",
            })
            continue

        hdf_path = Path(hdf_path)

        # Use the summary output: maximum water surface per mesh cell.
        max_ws = HdfResultsMesh.get_mesh_max_ws(hdf_path)
        if max_ws.empty:
            extraction_failures.append({
                "plan_number": plan_num,
                "land_cover": row["land_cover"],
                "n_value": row["n_value"],
                "reason": "empty_max_wse_summary",
            })
            continue

        coords = np.column_stack([max_ws.geometry.x, max_ws.geometry.y])
        tree = cKDTree(coords)
        dist, idx = tree.query([POI_X, POI_Y])
        nearest = max_ws.iloc[idx]
        max_wse = nearest["maximum_water_surface"]

        results.append({
            "land_cover": row["land_cover"],
            "n_value": row["n_value"],
            "plan_number": plan_num,
            "max_wse_ft": float(max_wse),
            "is_baseline": row["is_baseline"],
            "poi_cell_dist_ft": float(dist),
            "mesh_name": nearest.get("mesh_name", ""),
            "cell_id": int(nearest.get("cell_id", -1)),
        })

    except Exception as e:
        extraction_failures.append({
            "plan_number": plan_num,
            "land_cover": row["land_cover"],
            "n_value": row["n_value"],
            "reason": str(e),
        })

results_df = pd.DataFrame(results)
extraction_failures_df = pd.DataFrame(extraction_failures)
print()
print(f"Results extracted: {len(results_df)} of {len(registry_df)} plans")
if not extraction_failures_df.empty:
    display(extraction_failures_df)
    raise RuntimeError(
        f"WSE extraction failed for {len(extraction_failures_df)} registered plan(s)."
    )
if results_df.empty:
    raise RuntimeError("No WSE results were extracted from completed HDF files.")
print(f"POI cell distance: {results_df['poi_cell_dist_ft'].iloc[0]:.1f} ft")
print()
print(f"WSE range: {results_df['max_wse_ft'].min():.2f} - {results_df['max_wse_ft'].max():.2f} ft")
results_df.head(10)


# Build class-baseline references for comparative post-processing. Use an exact
# baseline plan when the sweep includes it; otherwise linearly interpolate between
# the bracketing Manning's n plans for that class. Endpoint nearest is used only
# when the tested range does not bracket the baseline.
def _reference_weights(lc_data, baseline_n):
    lc_data = lc_data.sort_values("n_value").reset_index(drop=True)
    exact = lc_data[np.isclose(lc_data["n_value"].astype(float), float(baseline_n), atol=1e-9)]
    if not exact.empty:
        ref = exact.iloc[0]
        return {
            "reference_type": "exact_class_baseline",
            "baseline_n": float(baseline_n),
            "low_plan_number": ref["plan_number"],
            "low_n": float(ref["n_value"]),
            "high_plan_number": ref["plan_number"],
            "high_n": float(ref["n_value"]),
            "weight_high": 0.0,
        }

    below = lc_data[lc_data["n_value"] < baseline_n]
    above = lc_data[lc_data["n_value"] > baseline_n]
    if not below.empty and not above.empty:
        lo = below.iloc[-1]
        hi = above.iloc[0]
        weight_high = (float(baseline_n) - float(lo["n_value"])) / (float(hi["n_value"]) - float(lo["n_value"]))
        return {
            "reference_type": "interpolated_class_baseline",
            "baseline_n": float(baseline_n),
            "low_plan_number": lo["plan_number"],
            "low_n": float(lo["n_value"]),
            "high_plan_number": hi["plan_number"],
            "high_n": float(hi["n_value"]),
            "weight_high": float(weight_high),
        }

    nearest = lc_data.iloc[(lc_data["n_value"].astype(float) - float(baseline_n)).abs().argmin()]
    return {
        "reference_type": "nearest_endpoint_baseline",
        "baseline_n": float(baseline_n),
        "low_plan_number": nearest["plan_number"],
        "low_n": float(nearest["n_value"]),
        "high_plan_number": nearest["plan_number"],
        "high_n": float(nearest["n_value"]),
        "weight_high": 0.0,
    }


reference_rows = []
comparison_rows = []
for lc_name, lc_data in results_df.groupby("land_cover", sort=True):
    baseline_n = float(range_table[str(lc_name)]["baseline"])
    ref = _reference_weights(lc_data, baseline_n)
    low_ref_row = lc_data[lc_data["plan_number"] == ref["low_plan_number"]].iloc[0]
    high_ref_row = lc_data[lc_data["plan_number"] == ref["high_plan_number"]].iloc[0]
    ref_wse = (
        float(low_ref_row["max_wse_ft"])
        + ref["weight_high"] * (float(high_ref_row["max_wse_ft"]) - float(low_ref_row["max_wse_ft"]))
    )
    ref.update({"reference_max_wse_ft": ref_wse})
    reference_rows.append({"land_cover": lc_name, **ref})

    for _, result_row in lc_data.iterrows():
        comparison_rows.append({
            **result_row.to_dict(),
            "baseline_n": baseline_n,
            "reference_type": ref["reference_type"],
            "reference_low_plan_number": ref["low_plan_number"],
            "reference_low_n": ref["low_n"],
            "reference_high_plan_number": ref["high_plan_number"],
            "reference_high_n": ref["high_n"],
            "reference_weight_high": ref["weight_high"],
            "reference_max_wse_ft": ref_wse,
            "max_wse_delta_ft": float(result_row["max_wse_ft"]) - ref_wse,
        })

reference_df = pd.DataFrame(reference_rows)
poi_comparison_df = pd.DataFrame(comparison_rows)

poi_class_rows = []
for lc_name, lc_data in poi_comparison_df.groupby("land_cover", sort=True):
    lc_data = lc_data.sort_values("n_value")
    low = lc_data.iloc[0]
    high = lc_data.iloc[-1]
    deltas = lc_data["max_wse_delta_ft"].to_numpy(dtype=float)
    poi_class_rows.append({
        "land_cover": lc_name,
        "baseline_n": float(low["baseline_n"]),
        "reference_type": low["reference_type"],
        "reference_low_plan_number": low["reference_low_plan_number"],
        "reference_low_n": low["reference_low_n"],
        "reference_high_plan_number": low["reference_high_plan_number"],
        "reference_high_n": low["reference_high_n"],
        "n_min": float(low["n_value"]),
        "n_max": float(high["n_value"]),
        "low_delta_ft": float(low["max_wse_delta_ft"]),
        "high_delta_ft": float(high["max_wse_delta_ft"]),
        "max_abs_delta_ft": float(np.nanmax(np.abs(deltas))),
        "delta_range_ft": float(np.nanmax(deltas) - np.nanmin(deltas)),
    })

poi_class_delta_df = pd.DataFrame(poi_class_rows).sort_values("max_abs_delta_ft", ascending=True)
print("\nClass reference methods:")
display(reference_df)


## Sensitivity Curves

Plot Manning’s n vs. Max WSE for each land cover class. Each subplot shows
how varying one land cover’s roughness (while holding others constant)
affects the water surface elevation at the POI.

In [ ]:
land_covers = poi_comparison_df["land_cover"].unique()
n_lc = len(land_covers)
ncols = min(3, n_lc)
nrows = int(np.ceil(n_lc / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
fig.suptitle(f"Baseline-Anchored Manning's n Sensitivity at {POI_LABEL}", fontsize=14, fontweight="bold")

for i, lc_name in enumerate(land_covers):
    ax = axes[i // ncols, i % ncols]
    lc_data = poi_comparison_df[poi_comparison_df["land_cover"] == lc_name].sort_values("n_value")
    ref = reference_df[reference_df["land_cover"] == lc_name].iloc[0]

    ax.plot(lc_data["n_value"], lc_data["max_wse_delta_ft"], "o-", color="steelblue", linewidth=2)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.axvline(ref["baseline_n"], color="red", linestyle="--", alpha=0.7, label="Baseline n")

    ax.set_title(f"{lc_name} ({ref['reference_type'].replace('_', ' ')})", fontsize=10)
    ax.set_xlabel("Manning's n")
    ax.set_ylabel("Max WSE delta from class baseline (ft)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused subplots
for j in range(n_lc, nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

plt.tight_layout()
plt.show()


## Tornado Diagram

The tornado diagram ranks land cover classes by their influence on WSE.
Wider bars indicate greater sensitivity — these are the parameters that
matter most for model calibration.

In [ ]:
# Tornado diagram using class-baseline-anchored deltas.
tornado_df = poi_class_delta_df.sort_values("max_abs_delta_ft", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(tornado_df) * 0.8)))
y_pos = range(len(tornado_df))

for i, (_, row) in enumerate(tornado_df.iterrows()):
    low_delta = row["low_delta_ft"]
    high_delta = row["high_delta_ft"]

    ax.barh(i, high_delta, left=0, height=0.6, color="steelblue", alpha=0.8, label="High n" if i == 0 else None)
    ax.barh(i, low_delta, left=0, height=0.6, color="coral", alpha=0.8, label="Low n" if i == 0 else None)

    ax.text(high_delta + (0.01 if high_delta >= 0 else -0.01), i, f"{high_delta:+.2f}",
            va="center", ha="left" if high_delta >= 0 else "right", fontsize=9)
    ax.text(low_delta + (0.01 if low_delta >= 0 else -0.01), i, f"{low_delta:+.2f}",
            va="center", ha="left" if low_delta >= 0 else "right", fontsize=9)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([
    f'{r["land_cover"]} (base n={r["baseline_n"]:.3f})'
    for _, r in tornado_df.iterrows()
])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Change in Max WSE from Class Baseline (ft)")
ax.set_title(f"Baseline-Anchored Manning's n Sensitivity\n{POI_LABEL}", fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

print("\n=== Baseline-Anchored Sensitivity Ranking (most to least influential) ===")
for _, row in tornado_df.iloc[::-1].iterrows():
    print(
        f'  {row["land_cover"]}: max |delta| = {row["max_abs_delta_ft"]:.3f} ft '
        f'(low {row["low_delta_ft"]:+.3f}, high {row["high_delta_ft"]:+.3f}; '
        f'n: {row["n_min"]:.3f} to {row["n_max"]:.3f}; {row["reference_type"]})'
    )


## Spatial and Global Delta Metrics

A single POI can miss hydraulic response elsewhere in the mesh. This section compares each land-cover sweep across the full 2D area and writes reviewable spatial outputs:

- cell-centered max-WSE range and high-minus-low delta grids
- mesh-wide max-inundation volume and wetted-area summaries
- face max-velocity range and high-minus-low delta grids
- first-wetting arrival-time range using a configurable depth threshold

The class-envelope metrics compare the tested min-to-max Manning's n range for each land-cover class, so they do not depend on a baseline plan being available.

In [ ]:
from ras_commander.hdf.HdfBase import HdfBase
import h5py

SPATIAL_DELTA_DEPTH_THRESHOLD = float(
    os.getenv("RAS_COMMANDER_711_ARRIVAL_DEPTH_THRESHOLD", "0.10")
)
spatial_delta_dir = project_folder / "oat_spatial_global_delta"
spatial_delta_dir.mkdir(exist_ok=True)

ACRE_FT = 43560.0
ACRE = 43560.0

first_plan_num = str(results_df.sort_values(["land_cover", "n_value"])["plan_number"].iloc[0]).zfill(2)
first_geom_hdf = RasCurrency.get_geom_hdf_path(first_plan_num, ras)
if first_geom_hdf is None:
    raise RuntimeError(f"Could not resolve geometry HDF for plan {first_plan_num}.")

available_meshes = HdfMesh.get_mesh_area_names(first_geom_hdf)
if not available_meshes:
    raise RuntimeError(f"No 2D flow areas found in geometry HDF: {first_geom_hdf}")

mesh_filter = os.getenv("RAS_COMMANDER_711_SPATIAL_MESHES", "").strip()
if mesh_filter:
    requested_meshes = [m.strip() for m in mesh_filter.split(",") if m.strip()]
    missing_meshes = sorted(set(requested_meshes) - set(available_meshes))
    if missing_meshes:
        raise RuntimeError(f"Requested spatial mesh(es) not found: {missing_meshes}")
    SPATIAL_DELTA_MESHES = requested_meshes
else:
    SPATIAL_DELTA_MESHES = available_meshes

print(f"Spatial/global deltas will use {len(SPATIAL_DELTA_MESHES)} 2D flow area(s): {SPATIAL_DELTA_MESHES}")


def _required_hdf_dataset(hdf_file, hdf_path):
    if hdf_path not in hdf_file:
        raise KeyError(f"Missing HDF dataset: {hdf_path}")
    return hdf_file[hdf_path]


def _volume_at_wse(max_wse, volume_info, volume_values):
    volumes = np.zeros(len(max_wse), dtype=float)
    for cell_id, (start, count) in enumerate(volume_info[:, :2].astype(int)):
        if count <= 0 or not np.isfinite(max_wse[cell_id]):
            volumes[cell_id] = np.nan
            continue
        table = volume_values[start : start + count]
        elevations = np.asarray(table[:, 0], dtype=float)
        cell_volumes = np.asarray(table[:, 1], dtype=float)
        volumes[cell_id] = np.interp(
            float(max_wse[cell_id]),
            elevations,
            cell_volumes,
            left=0.0,
            right=float(cell_volumes[-1]),
        )
    return volumes


def _arrival_hours(water_surface_ts, cell_min_elev, time_hours, depth_threshold):
    depth_ts = water_surface_ts - cell_min_elev.reshape(1, -1)
    wet = depth_ts > depth_threshold
    any_wet = wet.any(axis=0)
    first_idx = np.argmax(wet, axis=0)
    arrival = np.full(wet.shape[1], np.nan, dtype=float)
    arrival[any_wet] = time_hours[first_idx[any_wet]]
    return arrival


def _read_spatial_delta_arrays(hdf_path, mesh_name):
    hdf_path = Path(hdf_path)
    with h5py.File(hdf_path, "r") as hdf_file:
        summary_base = (
            "Results/Unsteady/Output/Output Blocks/Base Output/"
            f"Summary Output/2D Flow Areas/{mesh_name}"
        )
        ts_base = (
            "Results/Unsteady/Output/Output Blocks/Base Output/"
            f"Unsteady Time Series/2D Flow Areas/{mesh_name}"
        )
        geom_base = f"Geometry/2D Flow Areas/{mesh_name}"

        max_wse = np.asarray(
            _required_hdf_dataset(hdf_file, f"{summary_base}/Maximum Water Surface")[0, :],
            dtype=float,
        )
        max_face_velocity = np.asarray(
            _required_hdf_dataset(hdf_file, f"{summary_base}/Maximum Face Velocity")[0, :],
            dtype=float,
        )
        cell_min_elev = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Minimum Elevation")[()],
            dtype=float,
        )
        cell_area = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Surface Area")[()],
            dtype=float,
        )
        centers = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Center Coordinate")[()],
            dtype=float,
        )
        face_points = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/FacePoints Coordinate")[()],
            dtype=float,
        )
        face_point_indexes = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Faces FacePoint Indexes")[()],
            dtype=int,
        )
        volume_info = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Volume Elevation Info")[()],
            dtype=int,
        )
        volume_values = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Volume Elevation Values")[()],
            dtype=float,
        )
        time_days = np.asarray(
            _required_hdf_dataset(
                hdf_file,
                "Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/Time",
            )[()],
            dtype=float,
        )
        water_surface_ts = np.asarray(
            _required_hdf_dataset(hdf_file, f"{ts_base}/Water Surface")[()],
            dtype=float,
        )
        crs = HdfBase.get_projection(hdf_file)

    max_depth = np.maximum(max_wse - cell_min_elev, 0.0)
    return {
        "mesh_name": mesh_name,
        "max_wse": max_wse,
        "max_depth": max_depth,
        "max_face_velocity": max_face_velocity,
        "cell_min_elev": cell_min_elev,
        "cell_area": cell_area,
        "cell_volume": _volume_at_wse(max_wse, volume_info, volume_values),
        "arrival_hours": _arrival_hours(
            water_surface_ts,
            cell_min_elev,
            time_days * 24.0,
            SPATIAL_DELTA_DEPTH_THRESHOLD,
        ),
        "centers": centers,
        "face_points": face_points,
        "face_point_indexes": face_point_indexes,
        "volume_info": volume_info,
        "volume_values": volume_values,
        "crs": crs,
    }


def _assert_same_topology(left_arrays, right_arrays, label):
    for key in ["centers", "face_points", "face_point_indexes", "cell_area", "cell_min_elev"]:
        left = np.asarray(left_arrays[key])
        right = np.asarray(right_arrays[key])
        if left.shape != right.shape:
            raise RuntimeError(f"Topology mismatch for {label}: {key} shape {left.shape} != {right.shape}")
    if not np.allclose(left_arrays["centers"], right_arrays["centers"], rtol=0.0, atol=1e-3):
        raise RuntimeError(f"Topology mismatch for {label}: cell center coordinates differ.")
    if not np.array_equal(left_arrays["face_point_indexes"], right_arrays["face_point_indexes"]):
        raise RuntimeError(f"Topology mismatch for {label}: face point indexes differ.")
    if not np.allclose(left_arrays["face_points"], right_arrays["face_points"], rtol=0.0, atol=1e-3):
        raise RuntimeError(f"Topology mismatch for {label}: face point coordinates differ.")


def _interpolate_arrays(low_arrays, high_arrays, weight_high):
    _assert_same_topology(low_arrays, high_arrays, f"{low_arrays['mesh_name']} baseline interpolation")
    weight_high = float(weight_high)
    reference = {
        "mesh_name": low_arrays["mesh_name"],
        "max_wse": (
            np.asarray(low_arrays["max_wse"], dtype=float)
            + weight_high
            * (np.asarray(high_arrays["max_wse"], dtype=float) - np.asarray(low_arrays["max_wse"], dtype=float))
        ),
        "max_face_velocity": (
            np.asarray(low_arrays["max_face_velocity"], dtype=float)
            + weight_high
            * (
                np.asarray(high_arrays["max_face_velocity"], dtype=float)
                - np.asarray(low_arrays["max_face_velocity"], dtype=float)
            )
        ),
        "cell_min_elev": low_arrays["cell_min_elev"],
        "cell_area": low_arrays["cell_area"],
        "centers": low_arrays["centers"],
        "face_points": low_arrays["face_points"],
        "face_point_indexes": low_arrays["face_point_indexes"],
        "volume_info": low_arrays["volume_info"],
        "volume_values": low_arrays["volume_values"],
        "crs": low_arrays["crs"],
    }
    reference["max_depth"] = np.maximum(reference["max_wse"] - reference["cell_min_elev"], 0.0)
    reference["cell_volume"] = _volume_at_wse(
        reference["max_wse"],
        reference["volume_info"],
        reference["volume_values"],
    )
    reference["arrival_hours"] = (
        np.asarray(low_arrays["arrival_hours"], dtype=float)
        + weight_high
        * (
            np.asarray(high_arrays["arrival_hours"], dtype=float)
            - np.asarray(low_arrays["arrival_hours"], dtype=float)
        )
    )
    return reference


def _finite_delta_stats(values, prefix):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {
            f"{prefix}_abs_max": np.nan,
            f"{prefix}_abs_p95": np.nan,
            f"{prefix}_abs_mean": np.nan,
            f"{prefix}_count": 0,
            f"{prefix}_abs_ge_0_01": 0,
            f"{prefix}_abs_ge_0_10": 0,
        }
    abs_arr = np.abs(arr)
    return {
        f"{prefix}_abs_max": float(np.nanmax(abs_arr)),
        f"{prefix}_abs_p95": float(np.nanpercentile(abs_arr, 95)),
        f"{prefix}_abs_mean": float(np.nanmean(abs_arr)),
        f"{prefix}_count": int(arr.size),
        f"{prefix}_abs_ge_0_01": int(np.sum(abs_arr >= 0.01)),
        f"{prefix}_abs_ge_0_10": int(np.sum(abs_arr >= 0.10)),
    }


def _nan_range(stack):
    valid_count = np.isfinite(stack).sum(axis=0)
    result = np.full(stack.shape[1], np.nan, dtype=float)
    valid = valid_count >= 2
    if np.any(valid):
        result[valid] = np.nanmax(stack[:, valid], axis=0) - np.nanmin(stack[:, valid], axis=0)
    return result


def _wet_area_ac(arrays):
    wet = arrays["max_depth"] > SPATIAL_DELTA_DEPTH_THRESHOLD
    return float(np.nansum(arrays["cell_area"][wet]) / ACRE)


def _concat_finite(values):
    if not values:
        return np.array([], dtype=float)
    return np.concatenate([np.asarray(v, dtype=float).reshape(-1) for v in values])


def _total_volume_acft(mesh_arrays):
    return float(sum(np.nansum(a["cell_volume"]) for a in mesh_arrays.values()) / ACRE_FT)


def _total_wet_area_ac(mesh_arrays):
    return float(sum(_wet_area_ac(a) for a in mesh_arrays.values()))


spatial_arrays_by_plan = {}
spatial_plan_rows = []

for _, row in results_df.sort_values(["land_cover", "n_value"]).iterrows():
    plan_num = row["plan_number"]
    hdf_path = RasCurrency.get_plan_hdf_path(plan_num, ras)
    if hdf_path is None or not Path(hdf_path).exists():
        raise FileNotFoundError(f"Missing plan HDF for plan {plan_num}: {hdf_path}")
    hdf_path = Path(hdf_path)
    plan_mesh_arrays = {
        mesh_name: _read_spatial_delta_arrays(hdf_path, mesh_name)
        for mesh_name in SPATIAL_DELTA_MESHES
    }
    spatial_arrays_by_plan[plan_num] = plan_mesh_arrays
    arrivals = _concat_finite([a["arrival_hours"] for a in plan_mesh_arrays.values()])
    spatial_plan_rows.append({
        "plan_number": plan_num,
        "land_cover": row["land_cover"],
        "n_value": row["n_value"],
        "mesh_count": len(plan_mesh_arrays),
        "max_wse_global_ft": float(np.nanmax([np.nanmax(a["max_wse"]) for a in plan_mesh_arrays.values()])),
        "max_depth_global_ft": float(np.nanmax([np.nanmax(a["max_depth"]) for a in plan_mesh_arrays.values()])),
        "max_inundation_volume_acft": _total_volume_acft(plan_mesh_arrays),
        "wet_area_ac_gt_threshold": _total_wet_area_ac(plan_mesh_arrays),
        "max_face_velocity_model_units_per_s": float(
            np.nanmax([np.nanmax(np.abs(a["max_face_velocity"])) for a in plan_mesh_arrays.values()])
        ),
        "arrival_cell_count_gt_threshold": int(arrivals.size),
        "arrival_latest_hr": float(np.nanmax(arrivals)) if arrivals.size else np.nan,
    })

spatial_plan_metrics_df = pd.DataFrame(spatial_plan_rows)

spatial_class_rows = []
spatial_plan_delta_rows = []
spatial_cell_grids = {mesh_name: {} for mesh_name in SPATIAL_DELTA_MESHES}
spatial_face_grids = {mesh_name: {} for mesh_name in SPATIAL_DELTA_MESHES}

for _, ref in reference_df.iterrows():
    land_cover = str(ref["land_cover"])
    lc_data = results_df[results_df["land_cover"].astype(str) == land_cover].sort_values("n_value")
    plan_numbers = lc_data["plan_number"].tolist()
    low_plan_number = ref["low_plan_number"]
    high_plan_number = ref["high_plan_number"]

    plan_delta_work = {
        plan_num: {
            "wse": [],
            "velocity": [],
            "arrival": [],
            "volume_delta": 0.0,
            "wet_area_delta": 0.0,
            "newly_wet_cells": 0,
            "lost_wet_cells": 0,
        }
        for plan_num in plan_numbers
    }
    low_delta_wse_all = []
    high_delta_wse_all = []
    low_delta_velocity_all = []
    high_delta_velocity_all = []
    low_delta_arrival_all = []
    high_delta_arrival_all = []
    max_abs_wse_delta_all = []
    max_abs_velocity_delta_all = []
    max_abs_arrival_delta_all = []
    envelope_wse_range_all = []
    envelope_velocity_range_all = []
    envelope_arrival_range_all = []

    reference_volume = 0.0
    reference_wet_area = 0.0

    for mesh_name in SPATIAL_DELTA_MESHES:
        low_arrays = spatial_arrays_by_plan[low_plan_number][mesh_name]
        high_arrays = spatial_arrays_by_plan[high_plan_number][mesh_name]
        reference_arrays = _interpolate_arrays(low_arrays, high_arrays, ref["weight_high"])
        reference_volume += float(np.nansum(reference_arrays["cell_volume"]) / ACRE_FT)
        reference_wet_area += _wet_area_ac(reference_arrays)

        arrays_list = [spatial_arrays_by_plan[p][mesh_name] for p in plan_numbers]
        for plan_num, arrays in zip(plan_numbers, arrays_list):
            _assert_same_topology(reference_arrays, arrays, f"plan {plan_num} mesh {mesh_name}")

        wse_deltas = [arrays["max_wse"] - reference_arrays["max_wse"] for arrays in arrays_list]
        velocity_deltas = [arrays["max_face_velocity"] - reference_arrays["max_face_velocity"] for arrays in arrays_list]
        arrival_deltas = [arrays["arrival_hours"] - reference_arrays["arrival_hours"] for arrays in arrays_list]

        wse_stack = np.stack([a["max_wse"] for a in arrays_list], axis=0)
        velocity_stack = np.stack([a["max_face_velocity"] for a in arrays_list], axis=0)
        arrival_stack = np.stack([a["arrival_hours"] for a in arrays_list], axis=0)

        low_result_arrays = arrays_list[0]
        high_result_arrays = arrays_list[-1]
        low_delta_wse = low_result_arrays["max_wse"] - reference_arrays["max_wse"]
        high_delta_wse = high_result_arrays["max_wse"] - reference_arrays["max_wse"]
        low_delta_velocity = low_result_arrays["max_face_velocity"] - reference_arrays["max_face_velocity"]
        high_delta_velocity = high_result_arrays["max_face_velocity"] - reference_arrays["max_face_velocity"]
        low_delta_arrival = low_result_arrays["arrival_hours"] - reference_arrays["arrival_hours"]
        high_delta_arrival = high_result_arrays["arrival_hours"] - reference_arrays["arrival_hours"]

        max_abs_wse_delta = np.nanmax(np.abs(np.stack(wse_deltas, axis=0)), axis=0)
        max_abs_velocity_delta = np.nanmax(np.abs(np.stack(velocity_deltas, axis=0)), axis=0)
        max_abs_arrival_delta = np.nanmax(np.abs(np.stack(arrival_deltas, axis=0)), axis=0)

        spatial_cell_grids[mesh_name][land_cover] = {
            "wse_low_ref": low_delta_wse,
            "wse_high_ref": high_delta_wse,
            "wse_absmax_ref": max_abs_wse_delta,
            "arrival_low_ref": low_delta_arrival,
            "arrival_high_ref": high_delta_arrival,
            "arrival_absmax_ref": max_abs_arrival_delta,
        }
        spatial_face_grids[mesh_name][land_cover] = {
            "velocity_low_ref": low_delta_velocity,
            "velocity_high_ref": high_delta_velocity,
            "velocity_absmax_ref": max_abs_velocity_delta,
        }

        for plan_num, arrays, wse_delta, velocity_delta, arrival_delta in zip(
            plan_numbers,
            arrays_list,
            wse_deltas,
            velocity_deltas,
            arrival_deltas,
        ):
            work = plan_delta_work[plan_num]
            work["wse"].append(wse_delta)
            work["velocity"].append(velocity_delta)
            work["arrival"].append(arrival_delta)
            work["volume_delta"] += float(
                np.nansum(arrays["cell_volume"]) / ACRE_FT
                - np.nansum(reference_arrays["cell_volume"]) / ACRE_FT
            )
            work["wet_area_delta"] += float(_wet_area_ac(arrays) - _wet_area_ac(reference_arrays))
            plan_wet = np.isfinite(arrays["arrival_hours"])
            ref_wet = np.isfinite(reference_arrays["arrival_hours"])
            work["newly_wet_cells"] += int(np.count_nonzero(plan_wet & ~ref_wet))
            work["lost_wet_cells"] += int(np.count_nonzero(~plan_wet & ref_wet))

        low_delta_wse_all.append(low_delta_wse)
        high_delta_wse_all.append(high_delta_wse)
        low_delta_velocity_all.append(low_delta_velocity)
        high_delta_velocity_all.append(high_delta_velocity)
        low_delta_arrival_all.append(low_delta_arrival)
        high_delta_arrival_all.append(high_delta_arrival)
        max_abs_wse_delta_all.append(max_abs_wse_delta)
        max_abs_velocity_delta_all.append(max_abs_velocity_delta)
        max_abs_arrival_delta_all.append(max_abs_arrival_delta)
        envelope_wse_range_all.append(_nan_range(wse_stack))
        envelope_velocity_range_all.append(_nan_range(velocity_stack))
        envelope_arrival_range_all.append(_nan_range(arrival_stack))

    for _, lc_row in lc_data.iterrows():
        plan_num = lc_row["plan_number"]
        work = plan_delta_work[plan_num]
        plan_delta_row = {
            "plan_number": plan_num,
            "land_cover": land_cover,
            "n_value": float(lc_row["n_value"]),
            "baseline_n": float(ref["baseline_n"]),
            "reference_type": ref["reference_type"],
            "reference_low_plan_number": low_plan_number,
            "reference_low_n": ref["low_n"],
            "reference_high_plan_number": high_plan_number,
            "reference_high_n": ref["high_n"],
            "max_inundation_volume_delta_acft": float(work["volume_delta"]),
            "wet_area_delta_ac_gt_threshold": float(work["wet_area_delta"]),
            "newly_wet_cells_vs_reference": int(work["newly_wet_cells"]),
            "lost_wet_cells_vs_reference": int(work["lost_wet_cells"]),
        }
        plan_delta_row.update(_finite_delta_stats(_concat_finite(work["wse"]), "max_wse_delta_ft"))
        plan_delta_row.update(_finite_delta_stats(_concat_finite(work["velocity"]), "max_face_velocity_delta_model_units_per_s"))
        plan_delta_row.update(_finite_delta_stats(_concat_finite(work["arrival"]), "arrival_delta_hr"))
        spatial_plan_delta_rows.append(plan_delta_row)

    class_plan_delta = pd.DataFrame([
        {
            "plan_number": p,
            "n_value": float(lc_data.loc[lc_data["plan_number"] == p, "n_value"].iloc[0]),
            "volume_delta": plan_delta_work[p]["volume_delta"],
            "wet_area_delta": plan_delta_work[p]["wet_area_delta"],
        }
        for p in plan_numbers
    ]).sort_values("n_value")
    volume_deltas = class_plan_delta["volume_delta"].to_numpy(dtype=float)
    wet_area_deltas = class_plan_delta["wet_area_delta"].to_numpy(dtype=float)

    summary_row = {
        "land_cover": land_cover,
        "plans": len(plan_numbers),
        "mesh_count": len(SPATIAL_DELTA_MESHES),
        "baseline_n": float(ref["baseline_n"]),
        "reference_type": ref["reference_type"],
        "reference_low_plan_number": low_plan_number,
        "reference_low_n": ref["low_n"],
        "reference_high_plan_number": high_plan_number,
        "reference_high_n": ref["high_n"],
        "n_min": float(lc_data["n_value"].min()),
        "n_max": float(lc_data["n_value"].max()),
        "low_volume_delta_acft": float(volume_deltas[0]),
        "high_volume_delta_acft": float(volume_deltas[-1]),
        "max_abs_volume_delta_acft": float(np.nanmax(np.abs(volume_deltas))),
        "envelope_volume_range_acft": float(np.nanmax(volume_deltas) - np.nanmin(volume_deltas)),
        "low_wet_area_delta_ac": float(wet_area_deltas[0]),
        "high_wet_area_delta_ac": float(wet_area_deltas[-1]),
        "max_abs_wet_area_delta_ac": float(np.nanmax(np.abs(wet_area_deltas))),
        "envelope_wet_area_range_ac": float(np.nanmax(wet_area_deltas) - np.nanmin(wet_area_deltas)),
    }
    summary_row.update(_finite_delta_stats(_concat_finite(max_abs_wse_delta_all), "max_abs_wse_delta_from_reference_ft"))
    summary_row.update(_finite_delta_stats(_concat_finite(low_delta_wse_all), "low_wse_delta_from_reference_ft"))
    summary_row.update(_finite_delta_stats(_concat_finite(high_delta_wse_all), "high_wse_delta_from_reference_ft"))
    summary_row.update(_finite_delta_stats(_concat_finite(envelope_wse_range_all), "envelope_wse_range_ft"))
    summary_row.update(_finite_delta_stats(_concat_finite(max_abs_velocity_delta_all), "max_abs_face_velocity_delta_from_reference_model_units_per_s"))
    summary_row.update(_finite_delta_stats(_concat_finite(envelope_velocity_range_all), "envelope_face_velocity_range_model_units_per_s"))
    summary_row.update(_finite_delta_stats(_concat_finite(max_abs_arrival_delta_all), "max_abs_arrival_delta_from_reference_hr"))
    summary_row.update(_finite_delta_stats(_concat_finite(low_delta_arrival_all), "low_arrival_delta_from_reference_hr"))
    summary_row.update(_finite_delta_stats(_concat_finite(high_delta_arrival_all), "high_arrival_delta_from_reference_hr"))
    summary_row.update(_finite_delta_stats(_concat_finite(envelope_arrival_range_all), "envelope_arrival_range_hr"))
    spatial_class_rows.append(summary_row)

spatial_plan_delta_df = pd.DataFrame(spatial_plan_delta_rows)
spatial_class_delta_df = pd.DataFrame(spatial_class_rows).sort_values(
    "max_abs_volume_delta_acft",
    ascending=False,
)

spatial_plan_metrics_csv = spatial_delta_dir / "oat_spatial_plan_global_metrics.csv"
spatial_plan_delta_csv = spatial_delta_dir / "oat_spatial_plan_delta_vs_class_baseline.csv"
spatial_class_delta_csv = spatial_delta_dir / "oat_spatial_class_baseline_delta_summary.csv"
spatial_plan_metrics_df.to_csv(spatial_plan_metrics_csv, index=False)
spatial_plan_delta_df.to_csv(spatial_plan_delta_csv, index=False)
spatial_class_delta_df.to_csv(spatial_class_delta_csv, index=False)

try:
    import geopandas as gpd
    from shapely.geometry import LineString, Point

    first_plan_arrays_by_mesh = next(iter(spatial_arrays_by_plan.values()))
    cell_frames = []
    face_frames = []
    export_crs = None
    for mesh_name, reference_arrays in first_plan_arrays_by_mesh.items():
        export_crs = export_crs or reference_arrays["crs"]
        centers = reference_arrays["centers"]
        cell_df = pd.DataFrame({
            "mesh_name": mesh_name,
            "cell_id": np.arange(len(centers), dtype=int),
            "geometry": [Point(float(x), float(y)) for x, y in centers],
        })
        for land_cover, grids in spatial_cell_grids[mesh_name].items():
            safe_lc = str(land_cover).replace(" ", "_")
            cell_df[f"wse_low_{safe_lc}"] = grids["wse_low_ref"]
            cell_df[f"wse_high_{safe_lc}"] = grids["wse_high_ref"]
            cell_df[f"wse_abs_{safe_lc}"] = grids["wse_absmax_ref"]
            cell_df[f"arr_low_{safe_lc}"] = grids["arrival_low_ref"]
            cell_df[f"arr_high_{safe_lc}"] = grids["arrival_high_ref"]
            cell_df[f"arr_abs_{safe_lc}"] = grids["arrival_absmax_ref"]
        cell_frames.append(cell_df)

        face_points = reference_arrays["face_points"]
        face_geometries = []
        for fp0, fp1 in reference_arrays["face_point_indexes"]:
            face_geometries.append(LineString([
                (float(face_points[fp0, 0]), float(face_points[fp0, 1])),
                (float(face_points[fp1, 0]), float(face_points[fp1, 1])),
            ]))
        face_df = pd.DataFrame({
            "mesh_name": mesh_name,
            "face_id": np.arange(len(face_geometries), dtype=int),
            "geometry": face_geometries,
        })
        for land_cover, grids in spatial_face_grids[mesh_name].items():
            safe_lc = str(land_cover).replace(" ", "_")
            face_df[f"vel_low_{safe_lc}"] = grids["velocity_low_ref"]
            face_df[f"vel_high_{safe_lc}"] = grids["velocity_high_ref"]
            face_df[f"vel_abs_{safe_lc}"] = grids["velocity_absmax_ref"]
        face_frames.append(face_df)

    cell_delta_gpkg = spatial_delta_dir / "oat_cell_wse_arrival_delta_vs_class_baseline.gpkg"
    gpd.GeoDataFrame(pd.concat(cell_frames, ignore_index=True), geometry="geometry", crs=export_crs).to_file(
        cell_delta_gpkg,
        layer="cell_deltas",
        driver="GPKG",
    )

    face_delta_gpkg = spatial_delta_dir / "oat_face_velocity_delta_vs_class_baseline.gpkg"
    gpd.GeoDataFrame(pd.concat(face_frames, ignore_index=True), geometry="geometry", crs=export_crs).to_file(
        face_delta_gpkg,
        layer="face_deltas",
        driver="GPKG",
    )
    print(f"Cell delta grids: {cell_delta_gpkg}")
    print(f"Face velocity delta grids: {face_delta_gpkg}")
except Exception as exc:
    print(f"Spatial GeoPackage export skipped: {exc}")

print(f"Spatial/global plan metrics: {spatial_plan_metrics_csv}")
print(f"Spatial/global plan deltas: {spatial_plan_delta_csv}")
print(f"Spatial/global class summary: {spatial_class_delta_csv}")
display(spatial_class_delta_df[[
    "land_cover",
    "mesh_count",
    "baseline_n",
    "reference_type",
    "n_min",
    "n_max",
    "low_volume_delta_acft",
    "high_volume_delta_acft",
    "max_abs_volume_delta_acft",
    "low_wet_area_delta_ac",
    "high_wet_area_delta_ac",
    "max_abs_wse_delta_from_reference_ft_abs_p95",
    "max_abs_arrival_delta_from_reference_hr_abs_p95",
    "max_abs_face_velocity_delta_from_reference_model_units_per_s_abs_p95",
]])

## Export Results

Save the full results table and tornado summary as CSV files for
further analysis or reporting.

In [ ]:
# Full POI results table
results_csv = project_folder / "oat_sensitivity_results.csv"
results_df.to_csv(results_csv, index=False)
print(f"Full POI results: {results_csv}")

# Class-baseline POI comparison table
poi_comparison_csv = project_folder / "oat_sensitivity_poi_delta_vs_class_baseline.csv"
poi_comparison_df.to_csv(poi_comparison_csv, index=False)
print(f"POI class-baseline deltas: {poi_comparison_csv}")

# Tornado summary, now anchored to class baseline references
tornado_csv = project_folder / "oat_sensitivity_tornado.csv"
tornado_df.to_csv(tornado_csv, index=False)
print(f"Baseline-anchored tornado summary: {tornado_csv}")

# Spatial/global summaries
print(f"Spatial/global plan metrics: {spatial_plan_metrics_csv}")
print(f"Spatial/global plan deltas: {spatial_plan_delta_csv}")
print(f"Spatial/global class summary: {spatial_class_delta_csv}")

print(f"\nTotal plans analyzed: {len(results_df)}")
print(f"Land covers varied: {results_df['land_cover'].nunique()}")
print(f"Overall POI WSE range: {results_df['max_wse_ft'].max() - results_df['max_wse_ft'].min():.3f} ft")
print(f"Max class-baseline POI |delta|: {tornado_df['max_abs_delta_ft'].max():.3f} ft")


## Cleanup

Remove the extracted example project to free disk space.

In [ ]:
import shutil

example_projects_dir = project_folder.parent
if example_projects_dir.name == "example_projects" and example_projects_dir.exists():
    shutil.rmtree(example_projects_dir, ignore_errors=True)
    print(f"Cleaned up: {example_projects_dir}")
else:
    print(f"Skipping cleanup: {example_projects_dir}")